# Product tagger with similarities

Because the expected proportion of products in our dataset is low and we need as much products as possible lets to try to come up with a way to select the subset of training set that would have a higher percentage of products. Also we would like for our model to have a good precision so it would be good to be tag many examples in the situations where this tagging is hard. First we create embeddings of our scrapped textual examples using OpenAI's ADA 002

In [8]:
import numpy as np
import pandas as pd
import openai
import math
from config_book import furniture_list

In [9]:
openai.api_key = '###########'
EMBEDDING_MODEL = "text-embedding-ada-002"

In [10]:
def get_embedding(text: str, model: str = EMBEDDING_MODEL) -> List[float]:
    """
    Gets the embedding vector for a given text using the specified model.

    Args:
        text (str): The text to get the embedding for.
        model (str, optional): The model name for embedding. Default is "text-embedding-ada-002".

    Returns:
        List[float]: The embedding vector.

    """
    return openai.Embedding.create(input=[text], model=model)['data'][0]['embedding']

Since this needed to be done in step by step fashion we did this using the ada_embedder.py (src.label.ada_embedder) and now we just load the embeddings

In [11]:
df = pd.read_pickle('../data/ada_embeddings.pkl')

In [12]:
df.head()

,clean,ada_embedding
0,3 Drawer Chests,"[-0.009600184857845306, 0.001654650317505002, ..."
1,Eau Rouge Armchair,"[-0.008347862400114536, -0.009867465123534203,..."
2,Entryway & Mudroom,"[0.010682865977287292, 0.008147293701767921, -..."
3,Kenneth Nilson,"[-0.012051408179104328, -0.019119305536150932,..."
4,Dania,"[-0.009124724194407463, 0.003880592528730631, ..."


The idea is as follows. Lets get 50 most standard names of furniture and embed them as well (this imported at the first cell of this notebook). Now let's calculate similarities between our examples and all of this 50 vectors. We will also make an average vector out of our standard 50 furniture vectors and calculate similarities between our examples and this vector. So the idea is that vectors which are likely to be furniture product names need to have at least some level of similarity with this average vector (think of this vector as a vector that represents general notion of furniture). But on the other side we don't want that our examples be to close to the any of 50 standard furniture names because this would lead to the picking too much of the general furniture names and not particular products (for example having just 'wooden chairs' would be too close to 'chairs' which is probably scraped from some drop down menu on web page where you can select types of furniture. On the other hand Veridon Garden Chair would not be too close to chairs but on the other side similar enough to average furniture vector to be selected). So in the nest couple of cells we will calculate these similarities and then try to find the best thresholds

Let's try both euclidean and cosine similarity

In [13]:
def euclidean_similarity(df, embeddings_column, desired_vector):
    df_copy = df.copy()
    # Convert the embeddings column to a NumPy array
    embeddings_array = np.array(df_copy[embeddings_column].tolist())
    desired_vector_array = np.array(desired_vector)

    # Calculate the Euclidean distances
    distances = np.linalg.norm(embeddings_array - desired_vector_array, axis=1)

    # Calculate the similarities as the inverse of distances
    similarities = 1 / (1 + distances)

    # Add the similarities as a new column in the DataFrame
    df_copy['similarity'] = similarities

    return df_copy.similarity

In [14]:
def cosine_similarity(df, embeddings_column, desired_vector):
    df_copy = df.copy()
    # Convert the embeddings column to a NumPy array
    embeddings_array = np.array(df_copy[embeddings_column].tolist())
    desired_vector_array = np.array(desired_vector)

    # Calculate the cosine similarities
    similarities = np.dot(embeddings_array, desired_vector_array)

    # Add the similarities as a new column in the DataFrame
    df_copy['similarity'] = similarities

    return df_copy.similarity

In [ ]:
for furniture in furniture_list:
    emb = get_embedding(furniture, model=EMBEDDING_MODEL)
    df_emb[f'{furniture}_euc'] = euclidean_similarity(df_emb, 'ada_embedding', emb)
    df_emb[f'{furniture}_cos'] = cosine_similarity(df_emb, 'ada_embedding', emb)

df_emb.to_pickle('../data/similarities.pkl')

In [15]:
df = pd.read_pickle('../data/similarities.pkl')

In [16]:
df.head()

,clean,ada_embedding,Desk_euc,Desk_cos,Chair_euc,Chair_cos,Table_euc,Table_cos,Sofa_euc,Sofa_cos,...,Display cabinet_euc,Display cabinet_cos,Sideboard_euc,Sideboard_cos,Lounge chair_euc,Lounge chair_cos,Buffet_euc,Buffet_cos,Credenza desk_euc,Credenza desk_cos
0,3 Drawer Chests,"[-0.009600184857845306, 0.001654650317505002, ...",0.627515,0.823828,0.612736,0.800272,0.607625,0.791502,0.618653,0.810017,...,0.656127,0.862662,0.610107,0.795804,0.609207,0.794252,0.602361,0.782111,0.655236,0.861573
1,Eau Rouge Armchair,"[-0.008347862400114536, -0.009867465123534203,...",0.617648,0.808391,0.637967,0.838984,0.600940,0.779512,0.635550,0.835583,...,0.612893,0.800537,0.590949,0.760434,0.667201,0.875599,0.602947,0.783176,0.622063,0.815439
2,Entryway & Mudroom,"[0.010682865977287292, 0.008147293701767921, -...",0.620073,0.812291,0.611084,0.797475,0.608514,0.793052,0.618333,0.809500,...,0.624907,0.819856,0.611130,0.797552,0.618773,0.810209,0.598747,0.775446,0.632679,0.831463
3,Kenneth Nilson,"[-0.012051408179104328, -0.019119305536150932,...",0.595738,0.769757,0.603267,0.783754,0.590150,0.758846,0.585474,0.749356,...,0.587580,0.753671,0.583402,0.745043,0.592697,0.763876,0.592647,0.763778,0.587657,0.753828
4,Dania,"[-0.009124724194407463, 0.003880592528730631, ...",0.600226,0.778196,0.600979,0.779584,0.589548,0.757642,0.604173,0.785385,...,0.589272,0.757089,0.576662,0.730534,0.597896,0.773850,0.592508,0.763506,0.595460,0.769225


Lets create average vector for furniture type

In [17]:
sum_vec = []
for furniture in furniture_list:
    sum_vec.append(get_embedding(furniture, model='text-embedding-ada-002'))
    avg_vec = np.average(np.array(sum_vec), axis=0)

Lets add similarities with average vector

In [18]:
df['avg_euc'] = euclidean_similarity(df, 'ada_embedding', avg_vec)

/var/folders/yw/f9bv06657hd27p6dc_53ct000000gn/T/ipykernel_97092/3954473577.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['avg_euc'] = euclidean_similarity(df, 'ada_embedding', avg_vec)


In [19]:
df['avg_cos'] = cosine_similarity(df, 'ada_embedding', avg_vec)

/var/folders/yw/f9bv06657hd27p6dc_53ct000000gn/T/ipykernel_97092/789758034.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['avg_cos'] = cosine_similarity(df, 'ada_embedding', avg_vec)


In [20]:
df.describe()

,Desk_euc,Desk_cos,Chair_euc,Chair_cos,Table_euc,Table_cos,Sofa_euc,Sofa_cos,Bookshelf_euc,Bookshelf_cos,...,Sideboard_euc,Sideboard_cos,Lounge chair_euc,Lounge chair_cos,Buffet_euc,Buffet_cos,Credenza desk_euc,Credenza desk_cos,avg_euc,avg_cos
count,78664.000000,78664.000000,78664.000000,78664.000000,78664.000000,78664.000000,78664.000000,78664.000000,78664.000000,78664.000000,...,78664.000000,78664.000000,78664.000000,78664.000000,78664.000000,78664.000000,78664.000000,78664.000000,78664.000000,78664.000000
mean,0.603798,0.783331,0.605318,0.786083,0.598876,0.774464,0.606619,0.787517,0.604444,0.784511,...,0.588475,0.754176,0.605814,0.785734,0.594730,0.766839,0.605154,0.784767,0.651793,0.777824
std,0.014437,0.026211,0.014400,0.025627,0.013421,0.024888,0.018779,0.031858,0.014386,0.025953,...,0.013369,0.026601,0.020341,0.033872,0.011810,0.022969,0.019111,0.033460,0.019143,0.024773
min,0.408508,-0.048254,0.409374,-0.040766,0.408917,-0.044713,0.409201,-0.042256,0.410290,-0.032914,...,0.410061,-0.034871,0.408376,-0.049398,0.409354,-0.040941,0.410908,-0.027657,0.418983,-0.039275
25%,0.593955,0.766326,0.595924,0.770113,0.590501,0.759545,0.593885,0.766190,0.594661,0.767690,...,0.579762,0.737299,0.592569,0.763626,0.587740,0.753997,0.591224,0.760979,0.638229,0.761584
50%,0.601411,0.780377,0.603068,0.783394,0.597036,0.772228,0.603173,0.783584,0.603228,0.783684,...,0.586992,0.752472,0.601580,0.780686,0.593819,0.766062,0.600408,0.778533,0.648274,0.775051
75%,0.612353,0.799627,0.612300,0.799537,0.604426,0.785840,0.616066,0.805810,0.612859,0.800480,...,0.595862,0.769994,0.614275,0.802848,0.600768,0.779196,0.617314,0.807849,0.663601,0.793746
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.998372,0.999999,...,1.000000,1.000000,0.998618,0.999999,1.000000,1.000000,0.800319,0.968874,0.768145,0.876682


We will take cosine distance first because of we have a larger variation with this metric.

In [21]:
# Cosine columns without average vector

cos_col = [col for col in df.columns if (col.endswith('cos') and not col.startswith('avg_'))]

Since we have a lot of examples un order to get the best thresholds quick we will take a reasonably sized random sample with concrete threshold levels of similarities. The size is governed by the condition that we can use normal distribution approximation and that confidence intervals are reasonably narrow. 

In [22]:
# This function returns a sample satisfying the given thresholds

def similarity_sampler(df, text_columnn, upper, lower, sample_size):
    condition1 = (df[cos_col] < upper).all(axis=1)
    condition2 = df['avg_cos'] > lower
    result = condition1 & condition2
    out = df[result][text_columnn]
    print(f'Number of tokens satisfying the condition is {out.size} \n')
    return out.sample(sample_size, random_state=2023)

In [23]:
# This function helps us annotate products and informs us in the end about the percentage of positive samples,
# validity of normal distribution approximation, and width of confidence interval 

def tag_estimator(data):
    num_samples = data.size
    print(f'Data size is {num_samples} \n')
    pos_proportion = 0
    for example in data:
        try:
            annotation = input(f'Is "{example}" a product? (1 - yes, 0 - no)')
            annotation = int(annotation)
            assert annotation == 0 or annotation == 1
            pos_proportion = pos_proportion + annotation/num_samples
        except (AssertionError, ValueError) as e:
            num_samples = num_samples - 1
            print("Input only zeros and ones")
            continue
    if (num_samples*pos_proportion <= 5) or (num_samples*(1-pos_proportion) <= 5):
        print('Normal distribution approximation not valid')
        return
    # 95% confidence
    sam_var = 1.96 * math.sqrt((pos_proportion * (1-pos_proportion))/num_samples)
    print(f'Percentage of positive examples is {pos_proportion*100}% +/- {sam_var*100}% with 95% confidence')

Lets try 0.9, 0.8 and sample size of 100

In [18]:
data = similarity_sampler(df, 'clean', 0.9, 0.8, 100)
tag_estimator(data)

Data size is 100 

Is "Teak framed dining chair with woven PE seat " a product? (1 - yes, 0 - no)0
Is "Rattan 2 5 seater lounge with upholstered seat and " a product? (1 - yes, 0 - no)0
Is "Bedroom Furniture" a product? (1 - yes, 0 - no)0
Is "Includes 4 Spacious Drawers" a product? (1 - yes, 0 - no)0
Is "Headboard Grand - Bellus Furniture" a product? (1 - yes, 0 - no)1
Is "Whitewood Desk Apato Melbourne Japanese Furniture Richmond 3121" a product? (1 - yes, 0 - no)1
Is "DINING ROOMS" a product? (1 - yes, 0 - no)0
Is "Dimensions 95 25 W x 16 5 D x 14 H" a product? (1 - yes, 0 - no)0
Is "Small 2 Door 1 Drawer Sideboard" a product? (1 - yes, 0 - no)0
Is "Velvet Cushion Brown " a product? (1 - yes, 0 - no)0
Is "Mid Century Dressers & Chest of Drawers" a product? (1 - yes, 0 - no)0
Is "Essence Sideboard" a product? (1 - yes, 0 - no)0
Is "Reclaimed Bar Furniture" a product? (1 - yes, 0 - no)0
Is "Cybil armchair" a product? (1 - yes, 0 - no)1
Is "Fleur dining chair - white" a product? (1 - ye

Now we go lower to 0.85 and 0.75

In [19]:
data = similarity_sampler(df, 'clean', 0.85, 0.75, 100)
tag_estimator(data)

Data size is 100 

Is "91cm" a product? (1 - yes, 0 - no)0
Is "662896040241" a product? (1 - yes, 0 - no)0
Is "Pocket springs HR Foam 35 kg" a product? (1 - yes, 0 - no)0
Is " Sales " a product? (1 - yes, 0 - no)0
Is "Cymax" a product? (1 - yes, 0 - no)0
Is "Jul 18 2019" a product? (1 - yes, 0 - no)0
Is "French Polynesia" a product? (1 - yes, 0 - no)0
Is "info tove-kindt-larsen 1" a product? (1 - yes, 0 - no)
Input only zeros and ones
Is "Pebble wall ceilling" a product? (1 - yes, 0 - no)0
Is "1 969 00" a product? (1 - yes, 0 - no)0
Is "$2 600 00" a product? (1 - yes, 0 - no)0
Is "White - $640 00 AUD" a product? (1 - yes, 0 - no)0
Is "Belle Champagne Silver Finish" a product? (1 - yes, 0 - no)0
Is "Landscape Votive Sage Amber" a product? (1 - yes, 0 - no)1
Is "TAPER CANDLES" a product? (1 - yes, 0 - no)0
Is "Unique foot glides" a product? (1 - yes, 0 - no)0
Is "CHAIRS Chairs Contracts forms Forms Nissin Mokkou Seating" a product? (1 - yes, 0 - no)0
Is "00656237701007" a product? (1 - y

Let's refine with 0.87 and 0.8

In [20]:
data = similarity_sampler(df, 'clean', 0.87, 0.8, 100)
tag_estimator(data)

Data size is 100 

Is "Antique Small Oval Footstool" a product? (1 - yes, 0 - no)1
Is "Nest of Tables" a product? (1 - yes, 0 - no)0
Is "Tegmen Modern White Bed Frame " a product? (1 - yes, 0 - no)1
Is "Brinkley Double Chest Armoire" a product? (1 - yes, 0 - no)1
Is "8770 Accent Table" a product? (1 - yes, 0 - no)1
Is "Aiden 3-Seater Sofa" a product? (1 - yes, 0 - no)1
Is "Mirrored front and end panels " a product? (1 - yes, 0 - no)0
Is "Tray " a product? (1 - yes, 0 - no)0
Is "Arch console" a product? (1 - yes, 0 - no)1
Is "Wall Clock" a product? (1 - yes, 0 - no)0
Is "Jenny Crossback Armchair Antique Oak" a product? (1 - yes, 0 - no)1
Is "Apex Study Table - W1800" a product? (1 - yes, 0 - no)1
Is "Bertoia Molded Shell Side Chair - Stacking" a product? (1 - yes, 0 - no)1
Is "Vegas small extendable table" a product? (1 - yes, 0 - no)1
Is "Linear Steel Bench" a product? (1 - yes, 0 - no)0
Is "Rockville Office Set" a product? (1 - yes, 0 - no)1
Is "Main menu" a product? (1 - yes, 0 - no)

In the end 0.87 and 0.8 seem to be good enough

In [23]:
data = similarity_sampler(df, 'clean', 0.87, 0.8, 100)

Number of tokens satisfying the condition is 6024 



Now that we have a set with high proportion of products lets tag 1000 examples

In [1]:
def product_tagger(df, text_columnn, upper, lower):
    df_copy = df.copy()
    condition1 = (df_copy[cos_col] < upper).all(axis=1)
    condition2 = df_copy['avg_cos'] > lower
    result = condition1 & condition2
    df_copy = df_copy[result]
    print(f'Number of tokens satisfying the condition is {df_copy.shape[0]} \n')
    num_samples = df_copy.size
    i = 0
    for ind, example in df[result][text_columnn].items():
        try:
            annotation = input(f'Is "{example}" a product? (1 - yes, 0 - no)')
            annotation = int(annotation)
            assert annotation == 0 or annotation == 1
            df_copy.at[ind, 'tag'] = annotation
        except (AssertionError, ValueError) as e:
            num_samples = num_samples - 1
            print("Input only zeros and ones")
            continue
        i = i + 1
        if i%100 == 0:
            save = input(f'You have {i} examples. Do you want to save (y/n)?')
            if save == 'y':
                df_copy.to_csv(f'labeled_set_{100*lower}_{100*upper}.csv')
    print(df_copy)

Unfortenately I have accidentaly deleted the results of tagging but those can be seen in the <b> similarities.pdf </b> in this same folder. This is just for these 1000 examples the rest are here

In [2]:
product_tagger(df, 'clean', 0.87, 0.8) 

NameError: name 'df' is not defined

Now lets add some data that has high similarity with standard furniture names. We want our model to be able to predict good in this domain as well.

In [26]:
# Here we only need the condition that at least one similarity is higher than threshold

def product_tagger_adj_high(df, text_columnn, thrsh):
    df_copy = df.copy()
    condition = (df_copy[cos_col] > thrsh).any(axis=1)
    df_copy = df_copy[condition]
    print(f'Number of tokens satisfying the condition is {df_copy.shape[0]} \n')
    num_samples = df_copy.size
    i = 0
    for ind, example in df[condition][text_columnn].items():
        try:
            annotation = input(f'Is "{example}" a product? (1 - yes, 0 - no)')
            annotation = int(annotation)
            assert annotation == 0 or annotation == 1
            df_copy.at[ind, 'tag'] = annotation
        except (AssertionError, ValueError) as e:
            num_samples = num_samples - 1
            print("Input only zeros and ones")
            continue
        i = i + 1
        if i%100 == 0:
            save = input(f'You have {i} examples. Do you want to save (y/n)?')
            if save == 'y':
                df_copy.to_csv(f'labeled_set_{100*thrsh}.csv')
    print(df_copy)

In [52]:
product_tagger_adj_high(df, 'clean', 0.9)

Number of tokens satisfying the condition is 3123 

Is "Double Ottoman" a product? (1 - yes, 0 - no)0
Is "Cabinets & Cupboards" a product? (1 - yes, 0 - no)0
Is " SUMMIT Lounge Chair" a product? (1 - yes, 0 - no)1
Is "Puzzle sofa bed" a product? (1 - yes, 0 - no)0
Is "Lamp Table" a product? (1 - yes, 0 - no)0
Is "Sofa Tables" a product? (1 - yes, 0 - no)0
Is "TABLE" a product? (1 - yes, 0 - no)0
Is "Cabinets" a product? (1 - yes, 0 - no)0
Is "2 End Table" a product? (1 - yes, 0 - no)0
Is "cafe table" a product? (1 - yes, 0 - no)0
Is "Bookshelves & Bookcases" a product? (1 - yes, 0 - no)0
Is "Lido Barstool" a product? (1 - yes, 0 - no)1
Is "Ottomans & Poufs" a product? (1 - yes, 0 - no)0
Is "Dining room table" a product? (1 - yes, 0 - no)0
Is "Buffets & Credenza" a product? (1 - yes, 0 - no)0
Is "London table lamp" a product? (1 - yes, 0 - no)1
Is "Round tables" a product? (1 - yes, 0 - no)0
Is "Dining Tables Reduced" a product? (1 - yes, 0 - no)0
Is "Laurna Dining Table" a product? (1 

Is "3-seater Sofa" a product? (1 - yes, 0 - no)0
Is "Lucille Bedside Table" a product? (1 - yes, 0 - no)1
Is "Dressing table" a product? (1 - yes, 0 - no)0
Is "Writing Desk" a product? (1 - yes, 0 - no)0
Is "Combo Dresser" a product? (1 - yes, 0 - no)0
Is "Chardonnay Side table " a product? (1 - yes, 0 - no)1
Is "BAR STOOLS" a product? (1 - yes, 0 - no)0
Is "Storage & shelving" a product? (1 - yes, 0 - no)0
Is "Arc Dining Table " a product? (1 - yes, 0 - no)0
Is " TRI Stool" a product? (1 - yes, 0 - no)1
Is " Storage compartment" a product? (1 - yes, 0 - no)0
Is "Tv Units" a product? (1 - yes, 0 - no)0
Is "Mera Sofa" a product? (1 - yes, 0 - no)1
Is "Naruna Dining Table" a product? (1 - yes, 0 - no)1
Is "Breeze High Back Lounge Chair" a product? (1 - yes, 0 - no)1
Is "Bed Side Night Stand" a product? (1 - yes, 0 - no)0
Is "Dining Room Benches" a product? (1 - yes, 0 - no)0
Is "Office " a product? (1 - yes, 0 - no)0
Is "Stack Of Chairs Bar Stool" a product? (1 - yes, 0 - no)1
Is "dining

KeyboardInterrupt: Interrupted by user

In the end we want some examples from low similarity domain in order to have all kinds of examples seen by the model

In [53]:
product_tagger(df, 'clean', 0.8, 0)

Number of tokens satisfying the condition is 33402 

Is "Kenneth Nilson" a product? (1 - yes, 0 - no)0
Is " Dania " a product? (1 - yes, 0 - no)0
Is "$1 750 00" a product? (1 - yes, 0 - no)0
Is "Prestige Navy - $5 665 00 USD" a product? (1 - yes, 0 - no)0
Is "Niger USD $ " a product? (1 - yes, 0 - no)0
Is " Grofer HB has been added to your cart " a product? (1 - yes, 0 - no)0
Is "828 80" a product? (1 - yes, 0 - no)0
Is " $809 97" a product? (1 - yes, 0 - no)0
Is "$7 187 40" a product? (1 - yes, 0 - no)0
Is "Nov 8 2019" a product? (1 - yes, 0 - no)0
Is " 3 5 kg " a product? (1 - yes, 0 - no)0
Is "Lotty Damson" a product? (1 - yes, 0 - no)0
Is "24332" a product? (1 - yes, 0 - no)0
Is "Expand submenu ACCESSORIES" a product? (1 - yes, 0 - no)0
Is "Elligton Sectional" a product? (1 - yes, 0 - no)0
Is "Rich Navy Velvet" a product? (1 - yes, 0 - no)0
Is "$ 149 SGD" a product? (1 - yes, 0 - no)0
Is "173 00" a product? (1 - yes, 0 - no)0
Is "Trade Registration" a product? (1 - yes, 0 - no)0
Is

Is "jsprice" a product? (1 - yes, 0 - no)0
Is "contact adona in" a product? (1 - yes, 0 - no)0
Is "693009" a product? (1 - yes, 0 - no)0
Is "VGVCN8111-BLK-2pcs" a product? (1 - yes, 0 - no)0
Is "From $3330" a product? (1 - yes, 0 - no)0
Is "$5 577 53" a product? (1 - yes, 0 - no)0
Is "365 days FREE return policy " a product? (1 - yes, 0 - no)0
Is "12 - 26" a product? (1 - yes, 0 - no)0
Is "1 112 00" a product? (1 - yes, 0 - no)0
Is "color" a product? (1 - yes, 0 - no)0
Is "Robert Ogden" a product? (1 - yes, 0 - no)0
Is "59915" a product? (1 - yes, 0 - no)0
Is "for all UK parcel orders - next day & tracked " a product? (1 - yes, 0 - no)0
Is "RRP$540 26" a product? (1 - yes, 0 - no)0
Is "PERFORMANCE FABRICS" a product? (1 - yes, 0 - no)0
Is "AA Hanger - APATO" a product? (1 - yes, 0 - no)0
Is "139263" a product? (1 - yes, 0 - no)0
Is "3 sDIV Right or Left " a product? (1 - yes, 0 - no)0
Is "viewed cookie policy" a product? (1 - yes, 0 - no)0
Is "3 756 00" a product? (1 - yes, 0 - no)0
Is

KeyboardInterrupt: Interrupted by user

In [71]:
low = pd.read_csv('labeled_set_low.csv', index_col=[0], 
                  usecols = ['Unnamed: 0','clean', 'ada_embedding', 'tag'], nrows=200)
medium = pd.read_csv('labeled_set_medium.csv', index_col=[0], 
                  usecols = ['Unnamed: 0','clean', 'ada_embedding', 'tag'], nrows=1000)
high = pd.read_csv('labeled_set_high.csv', index_col=[0], 
                  usecols = ['Unnamed: 0','clean', 'ada_embedding', 'tag'], nrows=200)

Product percentage per distribution

In [72]:
(low.tag.sum()/(len(low)), medium.tag.sum()/(len(medium)), high.tag.sum()/(len(high)))

(0.015, 0.433, 0.265)

Now join

In [74]:
tagged_set = pd.concat([low, medium, high])

In [75]:
tagged_set

,clean,ada_embedding,tag
3,Kenneth Nilson,"[-0.012051408179104328, -0.019119305536150932,...",0.0
4,Dania,"[-0.009124724194407463, 0.003880592528730631, ...",0.0
5,$1 750 00,"[-0.00040550719131715596, 0.011903545819222927...",0.0
8,Prestige Navy - $5 665 00 USD,"[0.0036676963791251183, -0.008365873247385025,...",0.0
9,Niger USD $,"[-0.002789062215015292, -0.01558027882128954, ...",0.0
...,...,...,...
4860,Kink barstool,"[0.011534913443028927, -0.0043247416615486145,...",1.0
4916,Red Sofa,"[0.0008250460959970951, -0.0076873344369232655...",0.0
4985,Credenzas & Sideboards,"[0.0040961746126413345, -0.0027941998559981585...",0.0
5012,Nod Lounge Chair,"[-0.00554915564134717, -0.0031723971478641033,...",1.0


...and save

In [76]:
tagged_set.to_csv('tagged_set.csv')

Now lets make the validation set that will have the same distribution as the original data

In [50]:
def product_tagger_val(df, text_columnn, num_samples):
    df_copy = df.copy().sample(num_samples)
    print(f'Number of tokens satisfying the condition is {df_copy.shape[0]} \n')
    num_samples = df_copy.size
    pos_proportion = 0
    i = 0
    for ind, example in df_copy[text_columnn].items():
        try:
            annotation = input(f'Is "{example}" a product? (1 - yes, 0 - no)')
            annotation = int(annotation)
            assert annotation == 0 or annotation == 1
            df_copy.at[ind, 'tag'] = annotation
            pos_proportion = pos_proportion + annotation/num_samples
        except (AssertionError, ValueError) as e:
            num_samples = num_samples - 1
            print("Input only zeros and ones")
            continue
    # 95% confidence
    sam_var = 1.96 * math.sqrt((pos_proportion * (1-pos_proportion))/num_samples)
    print(f'Percentage of positive examples is {pos_proportion*100}% +/- {sam_var*100}% with 95% confidence')
    df_copy.to_csv(f'val_set_{num_samples}.csv')
    return

First lets exclude the training data

In [29]:
tagged_set = pd.read_pickle('../data/tagged_set_emb.pkl')

In [35]:
tag_for_join = tagged_set.drop(columns = ['clean', 'ada', 'bert'])

In [36]:
joined = df.merge(tag_for_join, how='outer', left_index=True, right_index=True, indicator=True)

In [42]:
not_tagged = joined[joined._merge == 'left_only'][['clean']]

Lets tag 500 examples

In [51]:
product_tagger_val(not_tagged, 'clean', 500)

Number of tokens satisfying the condition is 500 

Is "Transitional Black Leather Upholstery C king bed Phoenix by Coaster" a product? (1 - yes, 0 - no)1
Is "Checkout" a product? (1 - yes, 0 - no)0
Is " 2339 items " a product? (1 - yes, 0 - no)0
Is "Round 13 " a product? (1 - yes, 0 - no)0
Is "497 00" a product? (1 - yes, 0 - no)0
Is "Call " a product? (1 - yes, 0 - no)0
Is "$700" a product? (1 - yes, 0 - no)0
Is "78 682 00" a product? (1 - yes, 0 - no)0
Is "24429" a product? (1 - yes, 0 - no)0
Is "Plica Chair" a product? (1 - yes, 0 - no)1
Is "2 790" a product? (1 - yes, 0 - no)0
Is "Kaona abbreviation of Kaonashi from japanese " a product? (1 - yes, 0 - no)0
Is "hand-rubbed custom wood stains" a product? (1 - yes, 0 - no)0
Is "Bow Front Cases with Step Tops" a product? (1 - yes, 0 - no)0
Is "From $4834" a product? (1 - yes, 0 - no)0
Is "Order History" a product? (1 - yes, 0 - no)0
Is " - Coffee Tables Consoles Chests Sideboards - $339" a product? (1 - yes, 0 - no)0
Is "Luxury Ikat Cu

Is "Queen Size Bed 2 x Nightstands Dresser Mirror ONLY" a product? (1 - yes, 0 - no)0
Is "106 cm" a product? (1 - yes, 0 - no)0
Is "4 875 00" a product? (1 - yes, 0 - no)0
Is "Dec 6 2019" a product? (1 - yes, 0 - no)0
Is "Cocoon Lantern" a product? (1 - yes, 0 - no)1
Is "Feedback" a product? (1 - yes, 0 - no)0
Is " Exterior Fabric" a product? (1 - yes, 0 - no)0
Is "396 90" a product? (1 - yes, 0 - no)0
Is "457 00" a product? (1 - yes, 0 - no)0
Is " 55 0 kg" a product? (1 - yes, 0 - no)0
Is "W594 x D550 x H880 AH630 SH450 mm" a product? (1 - yes, 0 - no)0
Is "1 030 00" a product? (1 - yes, 0 - no)0
Is "Assembly Videos" a product? (1 - yes, 0 - no)0
Is "Grand Life Bergere Lounge Chair" a product? (1 - yes, 0 - no)1
Is "Byron Mini Side Table" a product? (1 - yes, 0 - no)1
Is "Shipping policy" a product? (1 - yes, 0 - no)0
Is "Continental Loft Plus" a product? (1 - yes, 0 - no)0
Is "Oak Brown Medea Platform Bedroom Set 3 PCS" a product? (1 - yes, 0 - no)1
Is "814 00" a product? (1 - yes, 0

Is " Mon - Sat 9am - 6pm" a product? (1 - yes, 0 - no)0
Is "TRADE ACCOUNTS" a product? (1 - yes, 0 - no)0
Is "From $6571" a product? (1 - yes, 0 - no)0
Is "Wool Rugs" a product? (1 - yes, 0 - no)0
Is "POUFS & OTTOMANS" a product? (1 - yes, 0 - no)0
Is "King Bed 142 W x 87 D x 61 H" a product? (1 - yes, 0 - no)0
Is " Sofa Beds" a product? (1 - yes, 0 - no)0
Is "$500 00 PICK UP " a product? (1 - yes, 0 - no)0
Is "Nov 6 2020" a product? (1 - yes, 0 - no)0
Is " c 2023 All rights reserved " a product? (1 - yes, 0 - no)0
Is "Kids Bed Safety Rails" a product? (1 - yes, 0 - no)0
Is "Living" a product? (1 - yes, 0 - no)0
Is "Atmosphere" a product? (1 - yes, 0 - no)0
Is "Contact information" a product? (1 - yes, 0 - no)0
Is "Uleg dining table" a product? (1 - yes, 0 - no)1
Is " SUBMIT" a product? (1 - yes, 0 - no)0
Is "Pictured is the Cave Creek L-Desk" a product? (1 - yes, 0 - no)0
Is "Very good service from start to end " a product? (1 - yes, 0 - no)0
Is "Systems with Storage" a product? (1 - 

Is "Discounted Cushion Covers" a product? (1 - yes, 0 - no)0
Is "$10 470" a product? (1 - yes, 0 - no)0
Is "Brand Ambassador " a product? (1 - yes, 0 - no)0
Is "30 Items " a product? (1 - yes, 0 - no)0
Is "From $1948" a product? (1 - yes, 0 - no)0
Is "KN02 - Swivel and Reclining High Back Lounge Chair" a product? (1 - yes, 0 - no)1
Is "Industrial Dining Tables" a product? (1 - yes, 0 - no)0
Is "Sofa set" a product? (1 - yes, 0 - no)0
Is "18 W x 16 D x 31 H" a product? (1 - yes, 0 - no)0
Is "ROUND BARSTOOL BY THONET" a product? (1 - yes, 0 - no)1
Is "Item is of superb quality Exceptionally pleasing packaging " a product? (1 - yes, 0 - no)0
Is "3-Piece Left-Arm Fasing Cornar Chaise Sectional" a product? (1 - yes, 0 - no)0
Is "Dark Walnut Finish" a product? (1 - yes, 0 - no)0
Is "Marrakech" a product? (1 - yes, 0 - no)0
Is "Lounge Stool - W550 x D390 x H415 mm" a product? (1 - yes, 0 - no)0
Is "Filter Fine Art" a product? (1 - yes, 0 - no)0
Is "Kenton Mill Kitchen Table" a product? (1 - y

Join ada embeddings to the validation set

In [77]:
val = pd.read_csv('../data/val_set_499.csv', index_col=0)

In [78]:
val.head()

,clean,tag
1116,Transitional Black Leather Upholstery C king b...,1.0
6293,Checkout,0.0
32879,2339 items,0.0
37637,Round 13,0.0
32213,497 00,0.0


In [86]:
val = val.merge(df[['ada_embedding']], how='inner', left_index=True, right_index=True)

In [90]:
val = val.dropna()

In [91]:
val.to_csv('../data/val_set_ada.csv')